# Module 3c: Ray Train - Distributed Training

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers Ray Train:
1. XGBoost distributed training
2. Scaling configuration
3. Checkpointing and results
4. Weather data prediction example

**Prerequisites**: `pip install "ray[train]" xgboost`

## Key Takeaways

- **Ray Train** provides distributed training for XGBoost, PyTorch, TensorFlow
- **ScalingConfig** controls workers and resources
- **Trainers** handle data distribution and result collection
- Direct integration with Ray Data for end-to-end pipelines

In [1]:
!pip install "ray[train]" xgboost

In [2]:
import ray
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Initialize Ray
if ray.is_initialized():
    ray.shutdown()

ray.init(num_cpus=4, logging_level="WARNING")
print(f"Ray version: {ray.__version__}")

Ray version: 2.55.1


---

## 1. XGBoost with Ray Train

### Connecting to DSC 232R

You've used XGBoost in Class13-15. Ray Train lets you scale XGBoost training across multiple workers.

In [3]:
# Generate sample regression data
X, y = make_regression(n_samples=10000, n_features=20, noise=0.1, random_state=42)

# Create pandas DataFrame
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

print(f"Dataset shape: {df.shape}")
print(f"Features: {feature_names[:5]}...")

Dataset shape: (10000, 21)
Features: ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4']...


In [4]:
# Convert to Ray Dataset
dataset = ray.data.from_pandas(df)

# Split into train/test
train_dataset, valid_dataset = dataset.train_test_split(test_size=0.2)

print(f"Train dataset: {train_dataset.count()} rows")
print(f"Valid dataset: {valid_dataset.count()} rows")

Train dataset: 8000 rows
Valid dataset: 2000 rows


### Basic XGBoostTrainer

In [10]:
import xgboost as xgb
import pandas as pd
from ray.train import ScalingConfig
from ray.train.xgboost import RayTrainReportCallback, XGBoostTrainer
from ray import train

# Define the training loop for each worker
def train_loop_per_worker(config):
    # Get the data shard for this worker
    train_dataset = train.get_dataset_shard("train")
    valid_dataset = train.get_dataset_shard("valid")

    # Convert DataIterator to pandas DataFrames by reading batches
    train_batches = list(train_dataset.iter_batches(batch_format="pandas"))
    valid_batches = list(valid_dataset.iter_batches(batch_format="pandas"))

    train_df = pd.concat(train_batches) if train_batches else pd.DataFrame()
    valid_df = pd.concat(valid_batches) if valid_batches else pd.DataFrame()

    # Separate features and labels
    label_col = config["label_column"]
    train_X, train_y = train_df.drop(columns=[label_col]), train_df[label_col]
    valid_X, valid_y = valid_df.drop(columns=[label_col]), valid_df[label_col]

    # Create DMatrix objects
    dtrain = xgb.DMatrix(train_X, label=train_y)
    dvalid = xgb.DMatrix(valid_X, label=valid_y)

    # Train the model
    results = {}
    xgb.train(
        config["params"],
        dtrain,
        num_boost_round=config["num_boost_round"],
        evals=[(dtrain, "train"), (dvalid, "valid")],
        evals_result=results,
        callbacks=[RayTrainReportCallback()]
    )

# Define trainer using XGBoostTrainer V2 API
trainer = XGBoostTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "label_column": "target",
        "num_boost_round": 50,
        "params": {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "max_depth": 6,
            "eta": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
        }
    },
    datasets={"train": train_dataset, "valid": valid_dataset},
    scaling_config=ScalingConfig(
        num_workers=2,  # Distributed across 2 workers
        use_gpu=False,
    ),
)

print("Trainer configured.")
print(f"  Scaling: {trainer.scaling_config}")

Trainer configured.
  Scaling: ScalingConfig(trainer_resources=None, num_workers=2, use_gpu=False, resources_per_worker=None, placement_strategy='PACK', accelerator_type=None, label_selector=None, use_tpu=False, topology=None, elastic_resize_monitor_interval_s=60.0)


In [11]:
# Train the model
result = trainer.fit()

print("\nTraining Results:")
print(f"  Final RMSE (valid): {result.metrics.get('valid-rmse', 'N/A')}")
print(f"  Checkpoint: {result.checkpoint}")

(TrainController pid=8407) Requesting resources: {'CPU': 1} * 2
(TrainController pid=8407) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller 

(pid=8656) Running Dataset train_7_0.: 0.00 row [00:00, ? row/s]

(pid=8656) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=8656) ✔️  Dataset train_7_0 execution finished in 0.16 seconds
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(pid=8657) Running Dataset valid_9_0.: 0.00 row [00:00, ? row/s]

(pid=8657) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(TrainController pid=8407) [18:25:55] [0]	train-rmse:187.45425	valid-rmse:191.01193
(RayTrainWorker pid=8523) Reporting training result 1: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-rmse': np.float64(187.4542503611731), 'valid-rmse': np.float64(191.011925882214)}), validation=False)
(RayTrainWorker pid=8523) Reporting training result 2: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-rmse': np.float64(175.60763551952832), 'valid-rmse': np.float64(179.24550777951126)}), validation=False)
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=8463) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=8556) [18:25:51] Task [xgboost.ray-rank=00000001]:be811ab0a1d882c6d23db35101000000 got rank 1
(SplitCoordinator pid=8657) Registered dataset logger for dataset val


Training Results:
  Final RMSE (valid): 50.09354895899355
  Checkpoint: Checkpoint(filesystem=local, path=/root/ray_results/ray_train_run-2026-05-19_18-25-20/checkpoint_2026-05-19_18-27-34.060905)


### Understanding ScalingConfig

In [12]:
# Different scaling configurations

scaling_configs = {
    "single_worker": ScalingConfig(
        num_workers=1,
        use_gpu=False,
    ),
    "multi_worker_cpu": ScalingConfig(
        num_workers=4,
        use_gpu=False,
        resources_per_worker={"CPU": 2},
    ),
    "multi_worker_gpu": ScalingConfig(
        num_workers=4,
        use_gpu=True,
        resources_per_worker={"GPU": 1},
    ),
}

print("Scaling Configuration Options:")
print("=" * 60)
for name, config in scaling_configs.items():
    print(f"\n{name}:")
    print(f"  Workers: {config.num_workers}")
    print(f"  Use GPU: {config.use_gpu}")
    print(f"  Resources: {config.resources_per_worker}")

Scaling Configuration Options:

single_worker:
  Workers: 1
  Use GPU: False
  Resources: None

multi_worker_cpu:
  Workers: 4
  Use GPU: False
  Resources: {'CPU': 2}

multi_worker_gpu:
  Workers: 4
  Use GPU: True
  Resources: {'GPU': 1}


---

## 2. Using the Trained Model

### Loading from Checkpoint

In [13]:
import xgboost as xgb

# Load model from checkpoint
checkpoint = result.checkpoint

# Get the booster
with checkpoint.as_directory() as checkpoint_dir:
    import os
    model_path = os.path.join(checkpoint_dir, "model.ubj")
    if os.path.exists(model_path):
        model = xgb.Booster()
        model.load_model(model_path)
        print("Model loaded from checkpoint.")
    else:
        print(f"Available files: {os.listdir(checkpoint_dir)}")

Model loaded from checkpoint.


### Making Predictions

In [16]:
import pandas as pd
import xgboost as xgb
import os

# The modern Ray API uses map_batches for distributed prediction
class PredictBatch:
    def __init__(self):
        # Load the predictor directly from the checkpoint on each worker
        with result.checkpoint.as_directory() as checkpoint_dir:
            model_path = os.path.join(checkpoint_dir, "model.ubj")
            self.model = xgb.Booster()
            self.model.load_model(model_path)

    def __call__(self, batch: pd.DataFrame) -> pd.DataFrame:
        # Convert to DMatrix, predict, and return results as a DataFrame
        dmatrix = xgb.DMatrix(batch)
        preds = self.model.predict(dmatrix)
        return pd.DataFrame({"prediction": preds})

# Prepare test data (remove target column)
test_features = valid_dataset.drop_columns(["target"])

# Make predictions using map_batches
predictions = test_features.map_batches(
    PredictBatch,
    concurrency=2,
    batch_format="pandas"
)

print("Sample predictions:")
print(predictions.take(5))

2026-05-19 18:30:41,036	WARNING util.py:642 -- The argument ``concurrency`` is deprecated in Ray 2.51. Please specify argument ``compute`` instead. For more information, see https://docs.ray.io/en/master/data/transforming-data.html#stateful-transforms.
2026-05-19 18:30:41,041	INFO logging.py:416 -- Registered dataset logger for dataset dataset_16_0
2026-05-19 18:30:41,045	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_16_0. Full logs are in /tmp/ray/session_2026-05-19_18-12-23_681714_4577/logs/ray-data
2026-05-19 18:30:41,045	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_16_0: InputDataBuffer[Input] -> ActorPoolMapOperator[MapBatches(drop_columns)->MapBatches(PredictBatch)] -> LimitOperator[limit=5]


Sample predictions:


2026-05-19 18:30:41,279	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_16_0 =======
2026-05-19 18:30:41,282	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 18:30:41,285	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store (pending: 2 CPU)
2026-05-19 18:30:41,286	INFO logging_progress.py:181 -- 
2026-05-19 18:30:41,290	INFO logging_progress.py:231 -- MapBatches(drop_columns)->MapBatches(PredictBatch): 0/1
2026-05-19 18:30:41,293	INFO logging_progress.py:233 --   Tasks: 0; Actors: 2 (running=0, restarting=0, pending=2); Queued blocks: 1 (0.0B); Resources: 0.0 CPU, 0.0B object store; [all objects local]
2026-05-19 18:30:41,295	INFO logging_progress.py:231 -- limit=5: 0/1
2026-05-19 18:30:41,298	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-05-19 18:30:41,302	INFO logging_progress.py:192 -- =============================================
202

[{'prediction': -79.40443420410156}, {'prediction': 289.8343811035156}, {'prediction': 160.39974975585938}, {'prediction': -125.88739013671875}, {'prediction': 115.04086303710938}]


---

## 3. Weather Data Prediction Example

Let's build a complete ML pipeline using weather data:

In [17]:
# Generate synthetic weather data
np.random.seed(42)
n_samples = 20000

# Features
temperature = np.random.uniform(-10, 40, n_samples)  # Celsius
humidity = np.random.uniform(20, 100, n_samples)  # Percent
pressure = np.random.uniform(980, 1040, n_samples)  # hPa
wind_speed = np.random.uniform(0, 30, n_samples)  # m/s
cloud_cover = np.random.uniform(0, 100, n_samples)  # Percent
month = np.random.randint(1, 13, n_samples)
hour = np.random.randint(0, 24, n_samples)

# Target: precipitation (mm) - synthetic relationship
precipitation = (
    0.5 * (humidity / 100) * (cloud_cover / 100) * 10 +
    0.3 * np.maximum(0, (humidity - 60) / 40) * 5 +
    np.random.exponential(0.5, n_samples)
)
precipitation = np.clip(precipitation, 0, 50)

weather_df = pd.DataFrame({
    "temperature": temperature,
    "humidity": humidity,
    "pressure": pressure,
    "wind_speed": wind_speed,
    "cloud_cover": cloud_cover,
    "month": month,
    "hour": hour,
    "precipitation": precipitation,
})

print(f"Weather dataset shape: {weather_df.shape}")
print(f"\nSample:")
weather_df.head()

Weather dataset shape: (20000, 8)

Sample:


,temperature,humidity,pressure,wind_speed,cloud_cover,month,hour,precipitation
0,8.727006,78.399865,997.934722,22.246656,81.816445,2,17,4.495243
1,37.535715,34.760960,985.689067,26.433056,14.526976,8,0,0.334335
2,26.599697,47.731176,987.581553,13.895396,94.646380,9,9,2.425595
3,19.932924,73.062451,990.840268,8.675362,84.322412,1,8,3.845287
4,-2.199068,58.567148,992.219200,9.565397,91.885330,6,19,2.845058


In [18]:
# Create Ray Dataset and preprocess
ds_weather = ray.data.from_pandas(weather_df)

# Feature engineering
def add_features(batch: pd.DataFrame) -> pd.DataFrame:
    """Add derived features."""
    result = batch.copy()

    # Dew point approximation
    result["dew_point"] = result["temperature"] - ((100 - result["humidity"]) / 5)

    # Is daytime (6am - 6pm)
    result["is_daytime"] = ((result["hour"] >= 6) & (result["hour"] < 18)).astype(int)

    # Season encoding (simple)
    result["is_winter"] = result["month"].isin([12, 1, 2]).astype(int)
    result["is_summer"] = result["month"].isin([6, 7, 8]).astype(int)

    return result

ds_processed = ds_weather.map_batches(add_features, batch_format="pandas")

print("Processed dataset columns:")
print(ds_processed.schema())

2026-05-19 18:30:54,365	INFO logging.py:416 -- Registered dataset logger for dataset dataset_19_0
2026-05-19 18:30:54,369	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_19_0. Full logs are in /tmp/ray/session_2026-05-19_18-12-23_681714_4577/logs/ray-data
2026-05-19 18:30:54,369	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_19_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(add_features)] -> LimitOperator[limit=1]
2026-05-19 18:30:54,393	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_19_0 =======
2026-05-19 18:30:54,395	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 18:30:54,396	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 18:30:54,397	INFO logging_progress.py:181 -- 
2026-05-19 18:30:54,398	INFO logging_progress.py:231 -- MapBatches(add_features): 0/1
2026-05-19 18:30:54,398	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queu

Processed dataset columns:


2026-05-19 18:30:54,648	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_19_0 execution finished in 0.28 seconds


Column         Type
------         ----
temperature    double
humidity       double
pressure       double
wind_speed     double
cloud_cover    double
month          int64
hour           int64
precipitation  double
dew_point      double
is_daytime     int64
is_winter      int64
is_summer      int64


In [19]:
# Split data
train_ds, valid_ds = ds_processed.train_test_split(test_size=0.2)

print(f"Training samples: {train_ds.count()}")
print(f"Validation samples: {valid_ds.count()}")

2026-05-19 18:30:54,664	INFO logging.py:416 -- Registered dataset logger for dataset dataset_20_0
2026-05-19 18:30:54,667	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_20_0. Full logs are in /tmp/ray/session_2026-05-19_18-12-23_681714_4577/logs/ray-data
2026-05-19 18:30:54,668	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_20_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(add_features)]
2026-05-19 18:30:54,687	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_20_0 =======
2026-05-19 18:30:54,689	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 18:30:54,690	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 18:30:54,691	INFO logging_progress.py:181 -- 
2026-05-19 18:30:54,692	INFO logging_progress.py:231 -- MapBatches(add_features): 0/1
2026-05-19 18:30:54,693	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.0B); Resou

Training samples: 16000
Validation samples: 4000


In [21]:
# Train XGBoost model for precipitation prediction
weather_trainer = XGBoostTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "label_column": "precipitation",
        "num_boost_round": 100,
        "params": {
            "objective": "reg:squarederror",
            "eval_metric": ["rmse", "mae"],
            "max_depth": 8,
            "eta": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "min_child_weight": 3,
        }
    },
    datasets={"train": train_ds, "valid": valid_ds},
    scaling_config=ScalingConfig(
        num_workers=2,
        use_gpu=False,
    ),
)

# Train
weather_result = weather_trainer.fit()

print("\nWeather Model Results:")
print(f"  Final RMSE: {weather_result.metrics.get('valid-rmse', 'N/A'):.4f}")
print(f"  Final MAE: {weather_result.metrics.get('valid-mae', 'N/A'):.4f}")

(TrainController pid=10591) Requesting resources: {'CPU': 1} * 2
(TrainController pid=10591) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2
(PlacementGroupCleaner pid=10655) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=10655) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=10655) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=10655) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=10655) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=10655) Failed to query Ray Train Con

(pid=10851) Running Dataset train_24_0.: 0.00 row [00:00, ? row/s]

(pid=10851) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=10851) Registered dataset logger for dataset train_24_0
(SplitCoordinator pid=10851) Starting execution of Dataset train_24_0. Full logs are in /tmp/ray/session_2026-05-19_18-12-23_681714_4577/logs/ray-data
(SplitCoordinator pid=10851) Execution plan of Dataset train_24_0: InputDataBuffer[Input] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=10851) ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(SplitCoordinator pid=10851) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext.get_current().enable_rich_progress_bars = True` and `ray.data.DataContext.get_current().use_ray_tqdm = False`.
(SplitCo

(pid=10852) Running Dataset valid_26_0.: 0.00 row [00:00, ? row/s]

(pid=10852) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(PlacementGroupCleaner pid=10655) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(TrainController pid=10591) [18:32:18] [0]	train-rmse:1.36470	train-mae:1.10026	valid-rmse:1.34550	valid-mae:1.08777
(RayTrainWorker pid=10711) Reporting training result 1: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-rmse': np.float64(1.3646961943832139), 'train-mae': np.float64(1.1002563985288143), 'valid-rmse': np.float64(1.3455033056340115), 'valid-mae': np.float64(1.0877745741307736)}), validation=False)
(RayTrainWorker pid=10711) Reporting training result 2: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-rmse': np.float64(1.2483853670931404), 'train-mae': np.float64(1.0059597424045206), 'valid-rmse': np.float64(1.2327646773865992), 'valid-mae': np.float64(0.9963156825602054)}), validation=False)
(TrainController pid=10591) [18:32:19] [1]	train-rmse:1.24839	train-mae:1.00596	valid-rmse:1.23276	valid-mae:0.9


Weather Model Results:
  Final RMSE: 0.5389
  Final MAE: 0.3966


---

## 4. Classification Example

In [22]:
# Generate classification data
X_cls, y_cls = make_classification(
    n_samples=5000,
    n_features=15,
    n_informative=10,
    n_classes=3,
    random_state=42
)

# Create DataFrame
cls_df = pd.DataFrame(X_cls, columns=[f"f{i}" for i in range(X_cls.shape[1])])
cls_df["label"] = y_cls

# Convert to Ray Dataset
cls_dataset = ray.data.from_pandas(cls_df)
cls_train, cls_valid = cls_dataset.train_test_split(test_size=0.2)

print(f"Classification dataset: {cls_df.shape}")
print(f"Classes: {np.unique(y_cls)}")

Classification dataset: (5000, 16)
Classes: [0 1 2]


In [23]:
# Train classifier
cls_trainer = XGBoostTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "label_column": "label",
        "num_boost_round": 50,
        "params": {
            "objective": "multi:softmax",
            "num_class": 3,
            "eval_metric": "mlogloss",
            "max_depth": 6,
            "eta": 0.1,
        }
    },
    datasets={"train": cls_train, "valid": cls_valid},
    scaling_config=ScalingConfig(num_workers=2),
)

cls_result = cls_trainer.fit()

print("\nClassification Results:")
print(f"  Final mlogloss: {cls_result.metrics.get('valid-mlogloss', 'N/A'):.4f}")

(TrainController pid=12441) Requesting resources: {'CPU': 1} * 2
(TrainController pid=12441) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Con

(pid=12692) Running Dataset train_31_0.: 0.00 row [00:00, ? row/s]

(pid=12692) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(SplitCoordinator pid=12692) Registered dataset logger for dataset train_31_0
(SplitCoordinator pid=12692) Starting execution of Dataset train_31_0. Full logs are in /tmp/ray/session_2026-05-19_18-12-23_681714_4577/logs/ray-data
(SplitCoordinator pid=12692) Execution plan of Dataset train_31_0: InputDataBuffer[Input] -> OutputSplitter[split(2, equal=True)]
(SplitCoordinator pid=12692) ⚠️  Ray's object store is configured to use only 42.9% of available memory (3.7GiB out of 8.6GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
(SplitCoordinator pid=12692) [dataset]: A new progress UI is available. To enable, set `ray.data.DataContext.get_current().enable_rich_progress_bars = True` and `ray.data.DataContext.get_current().use_ray_tqdm = False`.
(SplitCo

(pid=12691) Running Dataset valid_33_0.: 0.00 row [00:00, ? row/s]

(pid=12691) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(TrainController pid=12441) [18:37:47] [0]	train-mlogloss:1.02058	valid-mlogloss:1.03370
(RayTrainWorker pid=12560) Reporting training result 1: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-mlogloss': np.float64(1.0205764297246933), 'valid-mlogloss': np.float64(1.0336966528892517)}), validation=False)
(RayTrainWorker pid=12560) Reporting training result 2: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-mlogloss': np.float64(0.9544483678191901), 'valid-mlogloss': np.float64(0.9782783854603767)}), validation=False)
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(TrainController pid=12441) [18:37:49] [1]	train-mlogloss:0.95445	valid-mlogloss:0.97828
(PlacementGroupCleaner pid=12496) Failed to query Ray Train Contro


Classification Results:
  Final mlogloss: 0.3919


---

## 5. Exercise: Complete ML Pipeline

Build a complete pipeline that:
1. Loads data
2. Engineers features
3. Trains a model
4. Evaluates results

In [24]:
# Exercise: Build a pipeline to predict wind speed from other weather features

def build_wind_prediction_pipeline():
    """
    Build a pipeline to predict wind_speed from:
    - temperature
    - humidity
    - pressure
    - cloud_cover
    - month
    - hour

    Steps:
    1. Create Ray Dataset from weather_df (drop precipitation and wind_speed as features)
    2. Add feature engineering (pressure gradient proxy, etc.)
    3. Split into train/valid
    4. Train XGBoostTrainer with label_column="wind_speed"
    5. Return results
    """
    # Your code here
    pass

# result = build_wind_prediction_pipeline()
# print(f"Wind prediction RMSE: {result.metrics.get('valid-rmse', 'N/A')}")

In [25]:
# Solution

def build_wind_prediction_pipeline_solution():
    # Prepare data (use wind_speed as target, drop precipitation)
    wind_df = weather_df.drop(columns=["precipitation"])

    # Create Ray Dataset
    ds = ray.data.from_pandas(wind_df)

    # Feature engineering
    def engineer_features(batch: pd.DataFrame) -> pd.DataFrame:
        result = batch.copy()
        # Pressure deviation from mean (proxy for gradients)
        result["pressure_deviation"] = result["pressure"] - 1013.25
        # Temperature-humidity interaction
        result["temp_humidity"] = result["temperature"] * result["humidity"] / 100
        # Time features
        result["is_afternoon"] = ((result["hour"] >= 12) & (result["hour"] < 18)).astype(int)
        return result

    ds = ds.map_batches(engineer_features, batch_format="pandas")

    # Split
    train_ds, valid_ds = ds.train_test_split(test_size=0.2)

    # Train
    trainer = XGBoostTrainer(
        train_loop_per_worker=train_loop_per_worker,
        train_loop_config={
            "label_column": "wind_speed",
            "num_boost_round": 100,
            "params": {
                "objective": "reg:squarederror",
                "eval_metric": ["rmse", "mae"],
                "max_depth": 6,
                "eta": 0.1,
                "subsample": 0.8,
            }
        },
        datasets={"train": train_ds, "valid": valid_ds},
        scaling_config=ScalingConfig(num_workers=2),
    )

    return trainer.fit()

# Run solution
wind_result = build_wind_prediction_pipeline_solution()

print("\nWind Prediction Results:")
print(f"  RMSE: {wind_result.metrics.get('valid-rmse', 'N/A'):.4f}")
print(f"  MAE: {wind_result.metrics.get('valid-mae', 'N/A'):.4f}")

2026-05-19 18:40:16,662	INFO logging.py:416 -- Registered dataset logger for dataset dataset_37_0
2026-05-19 18:40:16,666	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_37_0. Full logs are in /tmp/ray/session_2026-05-19_18-12-23_681714_4577/logs/ray-data
2026-05-19 18:40:16,666	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_37_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(engineer_features)]
2026-05-19 18:40:16,688	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_37_0 =======
2026-05-19 18:40:16,690	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-19 18:40:16,691	INFO logging_progress.py:227 -- Active & requested resources: 0/4 CPU, 0.0B/1.8GiB object store
2026-05-19 18:40:16,693	INFO logging_progress.py:181 -- 
2026-05-19 18:40:16,695	INFO logging_progress.py:231 -- MapBatches(engineer_features): 0/1
2026-05-19 18:40:16,697	INFO logging_progress.py:233 --   Tasks: 1; Actors: 0; Queued blocks: 0 (0.

(pid=13835) Running Dataset train_41_0.: 0.00 row [00:00, ? row/s]

(pid=13835) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=13836) Running Dataset valid_43_0.: 0.00 row [00:00, ? row/s]

(pid=13836) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(TrainController pid=13582) [18:40:46] [0]	train-rmse:8.60058	train-mae:7.44553	valid-rmse:8.57698	valid-mae:7.43370
(RayTrainWorker pid=13702) Reporting training result 1: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-rmse': np.float64(8.600575682394316), 'train-mae': np.float64(7.445527257353067), 'valid-rmse': np.float64(8.576979369651843), 'valid-mae': np.float64(7.433696142315864)}), validation=False)
(RayTrainWorker pid=13702) Reporting training result 2: TrainingReport(checkpoint=None, metrics=OrderedDict({'train-rmse': np.float64(8.588060866422843), 'train-mae': np.float64(7.43442811730504), 'valid-rmse': np.float64(8.577828947564237), 'valid-mae': np.float64(7.434050400495529)}), validation=False)
(PlacementGroupCleaner pid=13638) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=13638) Failed to query Ray Train Controller actor state. State API may be temporarily unavaila


Wind Prediction Results:
  RMSE: 8.7120
  MAE: 7.5190


---

## 6. Comparison with Spark MLlib

| Aspect | Spark MLlib | Ray Train |
|--------|-------------|----------|
| Models | Limited built-in | XGBoost, PyTorch, TF, etc. |
| Data format | DataFrame | Ray Dataset |
| Distributed | Yes | Yes |
| GPU support | Limited | Native |
| Custom training | Limited | Full control |
| Best for | Simple models at scale | Complex ML pipelines |

---

## Summary

### Ray Train Key Concepts

1. **Trainers** - Pre-built for common frameworks
   - `XGBoostTrainer`, `LightGBMTrainer`
   - `TorchTrainer`, `TensorflowTrainer`

2. **ScalingConfig** - Control distributed training
   - `num_workers`: Number of training workers
   - `use_gpu`: Enable GPU training
   - `resources_per_worker`: CPU/GPU per worker

3. **Results** - Metrics and checkpoints
   - `result.metrics`: Training metrics
   - `result.checkpoint`: Saved model

### Pipeline Pattern

```
Data Source → Ray Data → Preprocessing → Train/Test Split → Ray Train → Metrics
```

### Next: Module 4

See `04_ray_spark_integration.md` for integrating Ray and Spark.

In [26]:
# Cleanup
ray.shutdown()
print("Ray shutdown complete.")

Ray shutdown complete.
